In [ ]:
import sys
sys.path.append('/home/qa/BlingTradePlatform/strategy/MomentumLearning/')
import Utils
import pandas as pd
import yfinance as yf
import requests
from ta.volatility import AverageTrueRange
import numpy as np
import json
import os
from MarketUniverse import MarketUniverse
from IndicatorCalculator import IndicatorAdder  # Fix the import
from DataFrameWindowFilter import DataWindowFilter
import mplfinance as mpf
import plotly.graph_objects as go
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource
from bokeh.io import output_file
from plotly.subplots import make_subplots
import sys
from backtesting import Backtest
sys.path.append('/home/qa/BlingTradePlatform/strategy')
from src.common.constants import *
from StrategyRepos.FirstBtStrategy import SmaCross
from ta.trend import CCIIndicator

class MomentumStrategy:
    def __init__(self) -> None:
        self.symboluniverse = MarketUniverse()
        self.inpdicatorCalculator = IndicatorAdder()
        self.dataframewindowfilter = DataWindowFilter


In [ ]:
import logging
from kiteconnect import KiteConnect
logging.basicConfig(level=logging.INFO)

kite = KiteConnect(api_key="lkk5m7jyvuhrg167")
data = kite.generate_session("4k45cpOL9H7aKCfwh2HfVlfsaW3pqb1A", api_secret="bpco0bsatv2yr56npqhv886yxqoed000")
kite.set_access_token(data["access_token"])


In [ ]:
kite.historical_data("424961", "2019-05-25 09:15:00", "2025-05-30 09:15:00", "hour", continuous=False, oi=False)

In [3]:
granularity = "1d"
# #periodInString = "28d"
# start_date_to_download = '2024-12-30'
# end_date_to_download = '2025-04-25'
# filter_start_date = '2024-01-01'
# filter_end_date = '2024-09-29'
# filter_start_time = '09:20::00'
# filter_end_time  = '15:30:00'
# STRATEGY_TYPE = SmaCross
# candleDownloadChart = False
CSV_FILE_PATH = GlobalConstants.historicMarketData_csv_dir
# if __name__ == "__main__":
strategy = MomentumStrategy()
current_date = Utils.get_current_date()
previous_date = Utils.calculate_dates(current_date, 1, 1)['past_date']

symbolListDataframe = strategy.symboluniverse.optionBackedEquityUniverse()
#symbolToSectorDict = strategy.symboluniverse.classify_stocks_to_sector()
#symbolListDataframe['Sector'] = symbolListDataframe['Symbol'].apply(lambda x :  symbolToSectorDict[x] if symbolToSectorDict.get(x) else "Others")
dirname = GlobalConstants.historicMarketData_dir
Utils.createDirectory(dirname)
Utils.save_df_to_csv(symbolListDataframe, dirname, 'equitySchema.csv')

Data is valid according to the schema /home/qa/BlingTradePlatform/schema/json/nfo_tradeable.json.
Data is valid according to the schema /home/qa/BlingTradePlatform/schema/json/nfo_tradeable.json.
Data is valid according to the schema /home/qa/BlingTradePlatform/schema/json/underlier.json.
Data is valid according to the schema /home/qa/BlingTradePlatform/schema/json/underlier.json.
Directory created: /home/qa/runtime/data/historicMktData/


In [ ]:
start_date_to_read= '2014-01-01'
end_date_to_read = '2025-03-28'
filterTypeDays = 6
print(symbolToSectorDict)
symbolCount = 0
allSymbologyDataFrame = []
for symbol in symbolList:
    print(symbolCount)
    symbolCount += 1
    #symbolDataframe = Utils.readCsv(symbol, granularity, start_date_to_download, dirname, ".csv",  periodInString)
    symbolDataframe = Utils.readCsv_from_startdate_enddate(symbol, granularity, start_date_to_read, end_date_to_read, dirname, ".csv")
    # from above symbolDataframe only keep the entries which are 6 rows apart from each other
    symbolDataframe = Utils.filter_dataframe_by_days(symbolDataframe, filterTypeDays)
    
    if symbolDataframe.empty:
        continue
    
    symbolDataframe = Utils.convert_yf_datetime_to_pandas_datetime(symbolDataframe)

    Utils.convert_datetime64_to_user_format(symbolDataframe['Date'], '%d-%m-%Y %H:%M:%S')
    symbolDataframe['Time'] = Utils.convert_datetime64_to_user_format(symbolDataframe['Date'], '%H:%M:%S')
    symbolDataframe = Utils.convert_yf_datetime_to_pandas_datetime(symbolDataframe)
    symbolDataframe['E10'] = symbolDataframe['Close'].ewm(span=10, adjust=False).mean()
    symbolDataframe['E21'] = symbolDataframe['Close'].ewm(span=21, adjust=False).mean()
    symbolDataframe['E50'] = symbolDataframe['Close'].ewm(span=50, adjust=False).mean()
    symbolDataframe.dropna(inplace=True)
    symbolDataframe['isCloseAbove10'] = symbolDataframe.apply(lambda x : 1 if x['Close'] > x['E10'] else -1, axis=1)
    symbolDataframe['isCloseAbove21'] = symbolDataframe.apply(lambda x : 1 if x['Close'] > x['E21'] else -1, axis=1)
    symbolDataframe['isCloseAbove50'] = symbolDataframe.apply(lambda x : 1 if x['Close'] > x['E50'] else -1, axis=1)
    symbolDataframe['isE10AboveE21'] = symbolDataframe.apply(lambda x : 1 if x['E10'] > x['E21'] else -1, axis=1)
    symbolDataframe['isE21AboveE50'] = symbolDataframe.apply(lambda x : 1 if x['E21'] > x['E50'] else -1, axis=1)
    symbolDataframe['isE10AboveE50'] = symbolDataframe.apply(lambda x : 1 if x['E10'] > x['E50'] else -1, axis=1)
    symbolDataframe['TotalValue'] = symbolDataframe['isCloseAbove10'] + symbolDataframe['isCloseAbove21'] + \
                                        symbolDataframe['isCloseAbove50'] + symbolDataframe['isE10AboveE21'] + \
                                            symbolDataframe['isE21AboveE50'] + symbolDataframe['isE10AboveE50']
    # ci = CCIIndicator(high=symbolDataframe['High'], low=symbolDataframe['Low'], close=symbolDataframe['Close'], window=3)
    # symbolDataframe['choppiness'] = ci.cci()
    atr = AverageTrueRange(high=symbolDataframe['High'], low=symbolDataframe['Low'], close=symbolDataframe['Close'], window=2).average_true_range()
    sum_atr = atr.rolling(window=2).sum()
    high_max = symbolDataframe['High'].rolling(window=2).max()
    low_min = symbolDataframe['Low'].rolling(window=2).min()
    ci = 100 * np.log10(sum_atr / (high_max - low_min)) / np.log10(2)
    symbolDataframe['choppiness'] = ci
    symbolDataframe['Symbol'] = symbol
    symbolDataframe['ReturnChange'] = ((symbolDataframe['Close'].shift(3) - symbolDataframe['Close'].shift(23))/symbolDataframe['Close'].shift(23)) * 100
    symbolDataframe.dropna(inplace=True)
    symbolDataframe['PRET'] = symbolDataframe.apply(lambda x : 1 if x['ReturnChange'] > 0 else -1, axis=1)
    symbolDataframe['Direction'] = symbolDataframe['Close'].diff()
    symbolDataframe['PositivePercentage'] = symbolDataframe['Direction'].shift(3).rolling(window=30).apply(lambda x: len([+1 for i in x if i > 0]))
    symbolDataframe['NegativePercentage'] = symbolDataframe['Direction'].shift(3).rolling(window=30).apply(lambda x: len([+1 for i in x if i < 0]))
    symbolDataframe['NoChange'] = symbolDataframe['Direction'].shift(3).rolling(window=30).apply(lambda x: len([+1 for i in x if i == 0]))
    symbolDataframe['Sector'] = symbolToSectorDict[symbol]
    symbolDataframe['FIPValue1'] = symbolDataframe['PRET'] * ((symbolDataframe['NegativePercentage']/30)- (symbolDataframe['PositivePercentage']/30))
    # for the above symbolDataframe create a new column called FIPValue whose value is symbolDataframe['NegativePercentage']/57 if PRET - -1 else symbolDataframe['PositivePercentage']/57
    symbolDataframe['FIPValue'] = symbolDataframe.apply(lambda x : x['NegativePercentage']/20 if x['PRET'] == -1 else x['PositivePercentage']/20, axis=1)
    #print(symbolDataframe.tail(50).loc[:, ['Close', 'Direction',  'PRET', 'PositivePercentage', 'NegativePercentage', 'FIPValue']])
    if symbolDataframe.empty:
        continue
    
    allSymbologyDataFrame.append(symbolDataframe)

# pp = (symbolDataframe.tail(50).loc[:, ['Symbol', 'Close', 'ReturnChange', 'PRET', 'Direction', 'PositivePercentage', 'NegativePercentage', 'FIPValue']])
# pp.reset_index(drop=True, inplace=True)
# pp
# print(symbolDataframe['FIPValue'].max())
# # print(symbolDataframe['FIPValue'].min())



In [ ]:
import pickle5 as pickle
# Save to file
# with open("dataframesJan2014ToMarch2025.pkl", "wb") as f:
#     pickle.dump(allSymbologyDataFrame, f)

#Load back
# with open("dataframesJan2014ToMarch2025Weekly.pkl", "rb") as f:
#     allSymbologyDataFrame = pickle.load(f)


## remove all the rows with FIPValue as NaN from allSymbologyDataFrame
for i in range(allSymbologyDataFrame.__len__()):
    allSymbologyDataFrame[i] = allSymbologyDataFrame[i][~allSymbologyDataFrame[i]['FIPValue'].isna()]
    allSymbologyDataFrame[i].reset_index(drop=True, inplace=True)

dates = allSymbologyDataFrame[0].iloc[0:]['Date'].to_list()
# AllDates = [date.strftime('%d-%m-%Y') for date in dates]
# AllDates
for i in range(allSymbologyDataFrame.__len__()):
    #allSymbologyDataFrame[i]['StrDate'] = allSymbologyDataFrame[i]['Date'].dt.strftime('%Y-%m-%d').copy()
    # Use .loc to avoid the warning about setting a value on a copy of a slice
    allSymbologyDataFrame[i].loc[:, 'StrDate'] = allSymbologyDataFrame[i]['Date'].dt.strftime('%Y-%m-%d')
    
    allSymbologyDataFrame[i].reset_index(drop=True, inplace=True)


# For allSymbologyDataFrame at index 0, get the row with the Date value as 'YYYY-MM-DD' format
date_to_find = '2021-01-01'
AllDates = [date.strftime('%Y-%m-%d') for date in dates]



# find the row with the date_to_find


In [ ]:
import numpy as np
print(AllDates)
totalDatesToRun = AllDates.__len__()
startDateForRun = '2014-12-10'
endDateTillRun = '2027-03-27'
forwardValue = 1 # indicates how many candles to move forward by....
totalScoreFilter = 5
# Find the next nearest startDateForRun entry value in AllDates and its index
nearest_start_date_index = next((i for i, date in enumerate(AllDates) if date >= startDateForRun), None)
nearest_start_date = AllDates[nearest_start_date_index] if nearest_start_date_index is not None else None

nearest_end_date_index = next((i for i, date in enumerate(AllDates) if date >= endDateTillRun), None)
nearest_end_date = AllDates[nearest_end_date_index] if nearest_end_date_index is not None else None

if nearest_end_date_index is None:
    nearest_end_date_index = totalDatesToRun - 1
    nearest_end_date = AllDates[nearest_end_date_index]
#print(nearest_start_date)
#print(nearest_end_date)
cashInPortfolio = 100000
portfolioList = []
symbolToPriceDict = {}
symbolPurchasedToQtyDict = {}
weeklyPortfolioValueDf = pd.DataFrame(columns=['Date', 'AccountValue'])
AdvanceDeclineDf = pd.DataFrame(columns=['Date', 'TotalAdvance'])


def getQtyPurchasedForSymbol(symbol):
    if symbolPurchasedToQtyDict.get(symbol) is None:
        return 0
    return symbolPurchasedToQtyDict[symbol]

def get_portfolio_value():
    totalPortfolioValue = 0
    for symbol in portfolioList:
        totalPortfolioValue += symbolToPriceDict[symbol] * getQtyPurchasedForSymbol(symbol)
    return totalPortfolioValue

def get_account_value():
    return (get_portfolio_value() + cashInPortfolio)

for runDate in range(nearest_start_date_index, nearest_end_date_index-1, forwardValue):
    current_run_date = AllDates[runDate]
    next_run_date = AllDates[runDate + 1]
    portfolioSymbolsToAdd = []
    currentSymbolToPriceDict= {}
    choppinessList = []
    advanceDecline = 0
    symbolDf1 = pd.DataFrame(columns=['Symbol', 'TotalScore', 'FIPScore'])

    for symbolDf  in allSymbologyDataFrame:
        filtered_df = Utils.filter_panda_df_in_range_constraint(symbolDf, 'StrDate', current_run_date, current_run_date)
                
        if filtered_df.empty:
            continue
        choppinessList.append(filtered_df['choppiness'].iloc[0])
        if filtered_df['Direction'].iloc[0] > 0:
            advanceDecline += 1
        
        symbolName = filtered_df['Symbol'].iloc[0]
        currentSymbolPrice = filtered_df['Close'].iloc[0]
        currentSymbolToPriceDict[symbolName] = currentSymbolPrice
        
        meanTotalScoreValue = filtered_df['TotalValue'].mean()
        
        FIPValue = filtered_df['FIPValue'].iloc[0]
        FIPValue1 = filtered_df['FIPValue1'].iloc[0]

        # append the symbolDf above to columns 'Symbol' mapped to symbolName, TotalScore to meanTotalScoreValue and FIPValue to FIPValue1
        dfx = pd.DataFrame({
            'Symbol': [symbolName],
            'TotalScore': [meanTotalScoreValue],
            'FIPScore' : [FIPValue1]
        })
        
        #symbolDf1 = pd.concat([symbolDf1, dfx], ignore_index=True)
        # Ensure that the DataFrame being concatenated is not empty
        # if not dfx.empty and not dfx.isna().all(axis=None):
        #     symbolDf1 = pd.concat([symbolDf1, dfx], ignore_index=True)
            # Ensure that the DataFrame being concatenated is not empty and does not contain all-NA rows
        if symbolDf1.empty:
            symbolDf1 = dfx
        else:
            if not dfx.empty and not dfx.isna().all(axis=None):
                symbolDf1 = pd.concat([symbolDf1, dfx.dropna(how='all')], ignore_index=True)

    advanceDeclineDfx = pd.DataFrame({
            'Date': [current_run_date],
            'TotalAdvance' : [(advanceDecline/224) * 100]
    })
    
    if AdvanceDeclineDf.empty:
        AdvanceDeclineDf = advanceDeclineDfx
    else:
        if not advanceDeclineDfx.empty and not advanceDeclineDfx.isna().all(axis=None):
            AdvanceDeclineDf = pd.concat([AdvanceDeclineDf, advanceDeclineDfx.dropna(how='all')], ignore_index=True)
    
    DatesList = nifyvix['Date'].to_list()
    nearest_next_index = next((i for i, date in enumerate(DatesList) if date >= current_run_date), None) - 1
    
    print("nifyvix date", nifyvix.iloc[nearest_next_index]['Date'])
    
    #print("weekly date {} choppiness value {}".format(current_run_date, mean_choppiness_value))
    # sort symbolDf1 first based on TotalScore descending then sort on FIPScore ascending
    symbolDf1 = symbolDf1.sort_values(by=['TotalScore', 'FIPScore'], ascending=[False, True])

    # from above symbolDf1 get the top 10 symbolNames
    print("nearest_next_index", nearest_next_index)
    if nifyvix.iloc[nearest_next_index]['percentile'] < 66:
        portfolioSymbolsToAdd = symbolDf1.head(6)['Symbol'].tolist()
    else:
        portfolioSymbolsToAdd = []
    #portfolioSymbolsToAdd.extend(top_symbols)

    # find common elements between portfolioList and portfolioSymbolsToAdd 
    commonSymbols = list(set(portfolioList) & set(portfolioSymbolsToAdd))
    symbolsToLiquidate = list(set(portfolioList) - set(commonSymbols))
    symbolsToAdd = list(set(portfolioSymbolsToAdd) - set(commonSymbols))
            
    for symbol in symbolsToLiquidate:
        cashInPortfolio += currentSymbolToPriceDict[symbol] * getQtyPurchasedForSymbol(symbol)
        cashInPortfolio -= currentSymbolToPriceDict[symbol] * getQtyPurchasedForSymbol(symbol) * 0.001
        portfolioList.remove(symbol)
    

    if symbolsToAdd.__len__() > 0:
        totalSymbols = len(symbolsToAdd)
        cashPerStock = cashInPortfolio/len(symbolsToAdd)
        # if cashPerStock > 5000:
        #     cashPerStock = 5000
        for symbol in symbolsToAdd:
            if totalSymbols <= 0:
                break
            cashPerStock = cashInPortfolio/totalSymbols
            stockQty = cashPerStock/currentSymbolToPriceDict[symbol]
            symbolPurchasedToQtyDict[symbol] = stockQty
            cashInPortfolio -= cashPerStock
            cashInPortfolio -= cashPerStock * 0.001
            totalSymbols -= 1                
            portfolioList.append(symbol)

    symbolToPriceDict = currentSymbolToPriceDict
    df2 = pd.DataFrame({
        'Date': [current_run_date],
        'AccountValue': [get_account_value()]
    })
    if weeklyPortfolioValueDf.empty:
        weeklyPortfolioValueDf = df2
    else:
        weeklyPortfolioValueDf = pd.concat([weeklyPortfolioValueDf, df2], ignore_index=True)
    print(portfolioSymbolsToAdd)
    print(nifyvix.iloc[nearest_next_index]['percentile'])
    print("Weekly date [" + current_run_date + "] " +  str(get_account_value()))

#print(weeklyPortfolioValueDf)


In [ ]:
(weeklyPortfolioValueDf)
weeklyPortfolioValueDf['Returns'] = weeklyPortfolioValueDf['AccountValue'].pct_change()

# Calculate Drawdown
weeklyPortfolioValueDf['CumulativeMax'] = weeklyPortfolioValueDf['AccountValue'].cummax()
weeklyPortfolioValueDf['Drawdown'] = (weeklyPortfolioValueDf['AccountValue'] - weeklyPortfolioValueDf['CumulativeMax']) / weeklyPortfolioValueDf['CumulativeMax']
weeklyPortfolioValueDf['RollingReturn'] = weeklyPortfolioValueDf['AccountValue'] / weeklyPortfolioValueDf['AccountValue'].iloc[0] - 1
weeklyPortfolioValueDf['CumulativeMeanReturn'] = weeklyPortfolioValueDf['Returns'].expanding().mean()
weeklyPortfolioValueDf['CumulativeStdReturn'] = weeklyPortfolioValueDf['Returns'].expanding().std()
weeklyPortfolioValueDf['SharpeRatio'] = weeklyPortfolioValueDf['CumulativeMeanReturn'] / weeklyPortfolioValueDf['CumulativeStdReturn']
weeklyPortfolioValueDf['ROMADRatio'] = ((weeklyPortfolioValueDf['AccountValue'] - 100000)/100000 )/ abs(weeklyPortfolioValueDf['Drawdown'])
# Calculate Sharpe Ratio (assumes risk-free rate = 0, and daily data)
#sharpe_ratio = weeklyPortfolioValueDf['Returns'].mean() / weeklyPortfolioValueDf['Returns'].std()

# Calculate Max Drawdown
#max_drawdown = weeklyPortfolioValueDf['Drawdown'].min()

# Total Return for ROMAD
#total_return = (weeklyPortfolioValueDf['AccountValue'].iloc[-1] / weeklyPortfolioValueDf['AccountValue'].iloc[0]) - 1
#romad = total_return / abs(max_drawdown) if max_drawdown != 0 else np.nan

# Add summary values to the DataFrame (repeated so it’s available per row)
#weeklyPortfolioValueDf['SharpeRatio'] = sharpe_ratio
#weeklyPortfolioValueDf['ROMAD'] = romad

# Clean-up
#weeklyPortfolioValueDf = weeklyPortfolioValueDf.drop(columns=['CumulativeMax'])
#weeklyPortfolioValueDf.plot(x='Date', y='AccountValue', kind='line', title='Weekly Portfolio Value Over Time', xlabel='Week', ylabel='Portfolio Value', rot=45)
# Create an interactive plot using plotly
import plotly.express as px

# Convert Week column to datetime for better plotting
weeklyPortfolioValueDf['Date'] = pd.to_datetime(weeklyPortfolioValueDf['Date'])

# Create an interactive line plot using plotly
fig = px.line(
    weeklyPortfolioValueDf,
    x='Date',
    y=['AccountValue'],
    labels={'value': 'Metrics', 'Date': 'Date'},
    title='Portfolio Metrics Over Time'
)

# Update layout for better readability
fig.update_layout(
    xaxis_title='Date',
    yaxis_title='Metrics',
    legend_title='Metrics',
    xaxis_tickangle=-45
)

# Show the plot
fig.show()

# create a plot for drawdown over date
drawdown_fig = px.line(
    weeklyPortfolioValueDf,
    x='Date',
    y=['Drawdown'],
    labels={'value': 'Drawdown', 'Date': 'Date'},
    title='Portfolio Drawdown Over Time'
)
drawdown_fig.update_layout(
    xaxis_title='Date',
    yaxis_title='Drawdown',
    legend_title='Metrics',
    xaxis_tickangle=-45
)
drawdown_fig.show()

# create similar plot for Sharpe Ratio and ROMADRatio

# Plot Sharpe Ratio over time
sharpe_ratio_fig = px.line(
    weeklyPortfolioValueDf,
    x='Date',
    y=['SharpeRatio'],
    labels={'value': 'Sharpe Ratio', 'Date': 'Date'},
    title='Sharpe Ratio Over Time'
)
sharpe_ratio_fig.update_layout(
    xaxis_title='Date',
    yaxis_title='Sharpe Ratio',
    legend_title='Metrics',
    xaxis_tickangle=-45
)
sharpe_ratio_fig.show()

# Plot ROMADRatio over time
romad_ratio_fig = px.line(
    weeklyPortfolioValueDf,
    x='Date',
    y=['ROMADRatio'],
    labels={'value': 'ROMAD Ratio', 'Date': 'Date'},
    title='ROMAD Ratio Over Time'
)
romad_ratio_fig.update_layout(
    xaxis_title='Date',
    yaxis_title='ROMAD Ratio',
    legend_title='Metrics',
    xaxis_tickangle=-45
)
romad_ratio_fig.show()


In [ ]:
import pandas as pd
import numpy as np

df = weeklyPortfolioValueDf.loc[:, ['Date', 'AccountValue']].copy()
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)

# Calculate CAGR
start_val = df['AccountValue'].iloc[0]
end_val = df['AccountValue'].iloc[-1]
years = (df.index[-1] - df.index[0]).days / 365.25
cagr = ((end_val / start_val) ** (1 / years) - 1) * 100

# Calculate average weekly return
df['WeeklyReturns'] = df['AccountValue'].pct_change()*100
weekly_returns = df['WeeklyReturns'].dropna()
avg_weekly_return = weekly_returns.mean() * 100

# Calculate percentage of positive and negative weekly returns
positive_weeks = (weekly_returns > 0).sum() / len(weekly_returns) * 100
negative_weeks = (weekly_returns < 0).sum() / len(weekly_returns) * 100

# Calculate drawdown compared to the last week

drawdown = df['AccountValue'].pct_change().apply(lambda x: x if x < 0 else 0)
df['Drawdown'] = drawdown

# Find the maximum drawdown period (consecutive weeks of negative returns)
negative_streaks = (weekly_returns < 0).astype(int).groupby((weekly_returns >= 0).cumsum()).cumsum()
max_drawdown_duration = negative_streaks.max()
# Find the weeks where the max_drawdown_duration happened and print them
drawdown_periods = (weekly_returns < 0).astype(int).groupby((weekly_returns >= 0).cumsum()).cumsum()
max_drawdown_weeks = drawdown_periods[drawdown_periods == max_drawdown_duration].index.tolist()

consec_neg = []
count = 0

for ret in df['WeeklyReturns']:
    if ret < 0:
        count += 1
    else:
        count = 0
    consec_neg.append(count)

# Assign the list as a new column
df['maxConsecutive'] = consec_neg
print(f"Weeks during the maximum drawdown duration: {max_drawdown_weeks}")
# Print the results
print(f"CAGR: {cagr:.2f}%")
print(f"Average Weekly Return: {avg_weekly_return:.2f}%")
print(f"Positive Weekly Returns: {positive_weeks:.2f}%")
print(f"Negative Weekly Returns: {negative_weeks:.2f}%")
print(df['maxConsecutive'].max())
max_index = df['maxConsecutive'].idxmax()
df.loc['2016-08-15 00:00:00' : '2016-11-15 00:00:00']

In [ ]:
from matplotlib import pyplot as plt

# Filter data for the period from 2015 to February 2017
filtered_df = AdvanceDeclineDf[(AdvanceDeclineDf['Date'] >= '2024-01-01') & (AdvanceDeclineDf['Date'] <= '2025-02-28')]

# Plotting
plt.figure(figsize=(10, 6))
plt.plot(filtered_df['Date'], filtered_df['TotalAdvance'], marker='o', label='Total Advance')

# Adding labels and title
plt.xlabel('Date')
plt.ylabel('Total Advance')
plt.title('Total Advance (2015 to February 2017)')
plt.grid(True)
plt.legend()

# Rotate x-axis labels for better readability
plt.xticks(rotation=45)

# Show the plot
plt.tight_layout()
plt.show()


In [ ]:
totalSymbolsConsidered = allSymbologyDataFrame.__len__()
print("Total Symbols Considered : ", totalSymbolsConsidered)

mapDateToTotalSymbolTotalValue = {}
for symbolDataframe in allSymbologyDataFrame:
    for index, row in symbolDataframe.iterrows():
        date = row['Date']
        totalValue = row['TotalValue']
        #print(date)
        #print(totalValue)
        if mapDateToTotalSymbolTotalValue.get(date):
            if mapDateToTotalSymbolTotalValue[date].get(totalValue):
                mapDateToTotalSymbolTotalValue[date][totalValue] += 1
            else:
                mapDateToTotalSymbolTotalValue[date][totalValue] = 1
        else:
            mapDateToTotalSymbolTotalValue[date] = {totalValue : 1}
# create dataframe sorted by date as index and -6 to +6 total number of symbols as columns
# for symbolDataframe in allSymbologyDataFrame:
#     symbolDataframe.reset_index(inplace=True)
#     print(symbolDataframe.head(10))
#     break
#print(mapDateToTotalSymbolTotalValue)
# Define all possible columns
columns = [-6, -5, -4, -3, -2, -1, 0, 1, 2, 3, 4, 5, 6]

# Convert dictionary to DataFrame
df = pd.DataFrame.from_dict(mapDateToTotalSymbolTotalValue, orient='index')

# Ensure all expected columns exist
df = df.reindex(columns=columns, fill_value=0)
df.dropna(inplace=True)
df['TotalNegative'] = df[-6] + df[-5] + df[-4] + df[-3] + df[-2] + df[-1]
df['TotalPositive'] = df[1] + df[2] + df[3] + df[4] + df[5] + df[6]
df['TotalAbove3'] =  df[4] + df[5] + df[6]
df['TotalBelow-3'] = df[-4] + df[-5] + df[-6]
df['NotGreat'] = df[-6] + df[-5] + df[-4] + df[-3] + df[-2] + df[-1] + df[0] + df[1] + df[2] + df[3]
df['PositiveMomentum'] = (df['TotalAbove3']/totalSymbolsConsidered)*100
df['NegativeMomentum'] = (df['NotGreat']/totalSymbolsConsidered)*100
# plot the above df dataframe into a plotting tool showing timedate on x-axis and PositiveMomentum and NegativeMomentum on y-axis as a bar chart

print(df.tail(30))


In [ ]:
import matplotlib.pyplot as plt

# Plotting the data
plt.figure(figsize=(14, 7))

# Plot PositiveMomentum
plt.bar(df.index, df['PositiveMomentum'], width=1.2, label='Positive Momentum', color='g', align='center')

# Plot NegativeMomentum
plt.bar(df.index, df['NegativeMomentum'], width=1.2, label='Negative Momentum', color='r', align='edge')

# Adding labels and title
plt.xlabel('Date')
plt.ylabel('Momentum (%)')
plt.title('Positive and Negative Momentum Over Time')
plt.legend()

# Rotate x-axis labels for better readability
plt.xticks(rotation=45)

# Set x-axis major locator to show more dates
plt.gca().xaxis.set_major_locator(plt.MaxNLocator(nbins=20))

# Show the plot
plt.tight_layout()
plt.show()

# make the above plot interactive using plotly
import plotly.express as px

# Create an interactive plot using plotly
fig = px.bar(df, x=df.index, y=['PositiveMomentum', 'NegativeMomentum'], 
             labels={'value': 'Momentum (%)', 'index': 'Date'}, 
             title='Positive and Negative Momentum Over Time')

# Update layout for better readability
fig.update_layout(barmode='overlay', xaxis_tickangle=-45)
fig.update_traces(marker=dict(line=dict(width=.5, color='DarkSlateGrey')))

# Show the plot
fig.show()

# Create a line plot for PositiveMomentum and NegativeMomentum using plotly
line_fig = px.line(
    df,
    x=df.index,
    y=['PositiveMomentum', 'NegativeMomentum'],
    labels={'value': 'Momentum (%)', 'index': 'Date'},
    title='Positive and Negative Momentum Over Time (Line Plot)'
)

# Update layout for better readability
line_fig.update_layout(
    xaxis_title='Date',
    yaxis_title='Momentum (%)',
    legend_title='Momentum Type',
    xaxis_tickangle=-45
)

# Show the plot
line_fig.show()


In [ ]:
import sys
sys.path.append('/home/qa/BlingTradePlatform/strategy/MomentumLearning/')
import Utils
import pandas as pd
import yfinance as yf
import requests
import json
import os
from MarketUniverse import MarketUniverse
from IndicatorCalculator import IndicatorAdder  # Fix the import
from DataFrameWindowFilter import DataWindowFilter
import mplfinance as mpf
import plotly.graph_objects as go
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource
from bokeh.io import output_file
from plotly.subplots import make_subplots
import sys
from backtesting import Backtest
sys.path.append('/home/qa/BlingTradePlatform/strategy')
from src.common.constants import *
from StrategyRepos.FirstBtStrategy import SmaCross
from itertools import product
from concurrent.futures import ProcessPoolExecutor

class MomentumStrategy:
    def __init__(self) -> None:
        self.symboluniverse = MarketUniverse()
        self.indicatorCalculator = IndicatorAdder()
        self.dataframewindowfilter = DataWindowFilter


granularity = "1d"
#periodInString = "28d"
start_date_to_download = '2024-12-30'
end_date_to_download = '2025-03-25'
filter_start_date = '2024-01-01'
filter_end_date = '2024-09-29'
filter_start_time = '09:20::00'
filter_end_time  = '15:30:00'
STRATEGY_TYPE = SmaCross
candleDownloadChart = False
CSV_FILE_PATH = GlobalConstants.historicMarketData_csv_dir
if __name__ == "__main__":
    strategy = MomentumStrategy()
    current_date = Utils.get_current_date()
    previous_date = Utils.calculate_dates(current_date, 1, 1)['past_date']
    
    symbolListDataframe = strategy.symboluniverse.optionBackedEquityUniverse()
    symbolToSectorDict = strategy.symboluniverse.classify_stocks_to_sector()
    symbolListDataframe['Sector'] = symbolListDataframe['Symbol'].apply(lambda x :  symbolToSectorDict[x] if symbolToSectorDict.get(x) else "Others")
    dirname = GlobalConstants.historicMarketData_dir
    Utils.createDirectory(dirname)
    Utils.save_df_to_csv(symbolListDataframe, dirname, 'equitySchema.csv')
    
    symbolList = Utils.panda_series_toList_converter(symbolListDataframe['Symbol'])

start_date_to_read= '2014-01-01'
end_date_to_read = '2025-03-28'
filterTypeDays = 5
symbolCount = 0

lookbackPeriod = 3
windowSize = 20

ewmWindow1 = 10
ewmWindow2 = 21
ewmWindow3 = 50
# Define the ranges for the parameters
# lookbackPeriod_range = range(3, 8)
# windowSize_range = range(20, 46)

# ewmWindow1_range = range(5, 21)
# ewmWindow2_range = range(21, 51)
# ewmWindow3_range = range(50, 71)

lookbackPeriod_range = range(3, 5)
windowSize_range = range(20, 22)

ewmWindow1_range = range(5, 7)
ewmWindow2_range = range(21, 23)
ewmWindow3_range = range(50, 52)

# Function to process a single combination of parameters
def process_combination(params):
    lookbackPeriod, windowSize, ewmWindow1, ewmWindow2, ewmWindow3 = params
    allSymbologyDataFrame = []
    for symbol in symbolList:
        symbolDataframe = Utils.readCsv_from_startdate_enddate(symbol, granularity, start_date_to_read, end_date_to_read, dirname, ".csv")
        symbolDataframe = Utils.filter_dataframe_by_days(symbolDataframe, filterTypeDays)
        
        if symbolDataframe.empty:
            continue
        symbolDataframe = Utils.convert_yf_datetime_to_pandas_datetime(symbolDataframe)
        Utils.convert_datetime64_to_user_format(symbolDataframe['Date'], '%d-%m-%Y %H:%M:%S')
        symbolDataframe['Time'] = Utils.convert_datetime64_to_user_format(symbolDataframe['Date'], '%H:%M:%S')
        symbolDataframe['E' + str(ewmWindow1)] = symbolDataframe['Close'].ewm(span=ewmWindow1, adjust=False).mean()
        symbolDataframe['E' + str(ewmWindow2)] = symbolDataframe['Close'].ewm(span=ewmWindow2, adjust=False).mean()
        symbolDataframe['E' + str(ewmWindow3)] = symbolDataframe['Close'].ewm(span=ewmWindow3, adjust=False).mean()
        symbolDataframe.dropna(inplace=True)
        symbolDataframe['isCloseAbove' + str(ewmWindow1)] = symbolDataframe.apply(lambda x: 1 if x['Close'] > x['E' + str(ewmWindow1)] else -1, axis=1)
        symbolDataframe['isCloseAbove' + str(ewmWindow2)] = symbolDataframe.apply(lambda x: 1 if x['Close'] > x['E' + str(ewmWindow2)] else -1, axis=1)
        symbolDataframe['isCloseAbove' + str(ewmWindow3)] = symbolDataframe.apply(lambda x: 1 if x['Close'] > x['E' + str(ewmWindow3)] else -1, axis=1)
        symbolDataframe['isE' + str(ewmWindow1) + 'AboveE' + str(ewmWindow2)] = symbolDataframe.apply(lambda x: 1 if x['E' + str(ewmWindow1)] > x['E' + str(ewmWindow2)] else -1, axis=1)
        symbolDataframe['isE' + str(ewmWindow2) + 'AboveE' + str(ewmWindow3)] = symbolDataframe.apply(lambda x: 1 if x['E' + str(ewmWindow2)] > x['E' + str(ewmWindow3)] else -1, axis=1)
        symbolDataframe['isE' + str(ewmWindow1) + 'AboveE' + str(ewmWindow3)] = symbolDataframe.apply(lambda x: 1 if x['E' + str(ewmWindow1)] > x['E' + str(ewmWindow3)] else -1, axis=1)
        symbolDataframe['TotalValue'] = symbolDataframe['isCloseAbove' + str(ewmWindow1)] + symbolDataframe['isCloseAbove' + str(ewmWindow2)] + \
                                        symbolDataframe['isCloseAbove' + str(ewmWindow3)] + symbolDataframe['isE' + str(ewmWindow1) + 'AboveE' + str(ewmWindow2)] + \
                                        symbolDataframe['isE' + str(ewmWindow2) + 'AboveE' + str(ewmWindow3)] + symbolDataframe['isE' + str(ewmWindow1) + 'AboveE' + str(ewmWindow3)]
        symbolDataframe['Symbol'] = symbol
        symbolDataframe['ReturnChange'] = ((symbolDataframe['Close'].shift(lookbackPeriod) - symbolDataframe['Close'].shift(windowSize + lookbackPeriod)) / symbolDataframe['Close'].shift(windowSize + lookbackPeriod)) * 100
        symbolDataframe.dropna(inplace=True)
        symbolDataframe['PRET'] = symbolDataframe.apply(lambda x: 1 if x['ReturnChange'] > 0 else -1, axis=1)
        symbolDataframe['Direction'] = symbolDataframe['Close'].diff()
        symbolDataframe['PositivePercentage'] = symbolDataframe['Direction'].shift(lookbackPeriod).rolling(window=windowSize).apply(lambda x: len([+1 for i in x if i > 0]))
        symbolDataframe['NegativePercentage'] = symbolDataframe['Direction'].shift(lookbackPeriod).rolling(window=windowSize).apply(lambda x: len([+1 for i in x if i < 0]))
        symbolDataframe['NoChange'] = symbolDataframe['Direction'].shift(lookbackPeriod).rolling(window=windowSize).apply(lambda x: len([+1 for i in x if i == 0]))
        symbolDataframe['Sector'] = symbolToSectorDict[symbol]
        symbolDataframe['FIPValue'] = symbolDataframe['PRET'] * ((symbolDataframe['NegativePercentage'] / windowSize) - (symbolDataframe['PositivePercentage'] / windowSize))
        symbolDataframe.dropna(inplace=True)
        if symbolDataframe.empty:
            continue
        allSymbologyDataFrame.append(symbolDataframe)
    return allSymbologyDataFrame

# Generate all combinations of parameters
parameter_combinations = list(product(lookbackPeriod_range, windowSize_range, ewmWindow1_range, ewmWindow2_range, ewmWindow3_range))

# Use ProcessPoolExecutor to parallelize the processing
with ProcessPoolExecutor(max_workers=15) as executor:
    #results = list(executor.map(lambda params: (params, process_combination(params)), parameter_combinations))
    results = list(executor.map(process_combination, parameter_combinations))

# AllDates = []
# for result in results:
#     for i in range(result.__len__()):
#         result[i] = result[i][~result[i]['FIPValue'].isna()]
#         result[i].reset_index(drop=True, inplace=True)

#     dates = result[0].iloc[0:]['Date'].to_list()
#     # AllDates = [date.strftime('%d-%m-%Y') for date in dates]
#     # AllDates
#     for i in range(result.__len__()):
#         result[i]['StrDate'] = result[i]['Date'].dt.strftime('%Y-%m-%d')
#         result[i].reset_index(drop=True, inplace=True)


#     # For allSymbologyDataFrame at index 0, get the row with the Date value as 'YYYY-MM-DD' format
#     date_to_find = '2021-01-01'
#     AllDates = [date.strftime('%Y-%m-%d') for date in dates]
#     break


In [ ]:
import sys
sys.path.append('/home/qa/BlingTradePlatform/strategy/MomentumLearning/')
import Utils
import pandas as pd
import yfinance as yf
import requests
import json
import os
from MarketUniverse import MarketUniverse
from IndicatorCalculator import IndicatorAdder  # Fix the import
from DataFrameWindowFilter import DataWindowFilter
import mplfinance as mpf
import plotly.graph_objects as go
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource
from bokeh.io import output_file
from plotly.subplots import make_subplots
import sys
from backtesting import Backtest
sys.path.append('/home/qa/BlingTradePlatform/strategy')
from src.common.constants import *
from StrategyRepos.FirstBtStrategy import SmaCross
from itertools import product
from concurrent.futures import ProcessPoolExecutor

class MomentumStrategy:
    def __init__(self) -> None:
        self.symboluniverse = MarketUniverse()
        self.indicatorCalculator = IndicatorAdder()
        self.dataframewindowfilter = DataWindowFilter
        
granularity = "1d"
#periodInString = "28d"
start_date_to_download = '2024-12-30'
end_date_to_download = '2025-04-25'
filter_start_date = '2024-01-01'
filter_end_date = '2024-09-29'
filter_start_time = '09:20::00'
filter_end_time  = '15:30:00'
STRATEGY_TYPE = SmaCross
candleDownloadChart = False
CSV_FILE_PATH = GlobalConstants.historicMarketData_csv_dir
if __name__ == "__main__":
    strategy = MomentumStrategy()
    current_date = Utils.get_current_date()
    previous_date = Utils.calculate_dates(current_date, 1, 1)['past_date']
    
    symbolListDataframe = strategy.symboluniverse.optionBackedEquityUniverse()
    symbolToSectorDict = strategy.symboluniverse.classify_stocks_to_sector()
    symbolListDataframe['Sector'] = symbolListDataframe['Symbol'].apply(lambda x :  symbolToSectorDict[x] if symbolToSectorDict.get(x) else "Others")
    dirname = GlobalConstants.historicMarketData_dir
    Utils.createDirectory(dirname)
    Utils.save_df_to_csv(symbolListDataframe, dirname, 'equitySchema.csv')
    
    symbolList = Utils.panda_series_toList_converter(symbolListDataframe['Symbol'])


start_date_to_read= '2014-01-01'
end_date_to_read = '2025-04-25'
filterTypeDays = 6
symbolCount = 0
allSymbologyDataFrame = []
#Utils.download_Mkt_Data_For_Symbols_Without_Start_And_EndDate_With_Granularity_And_SaveFilesToDirectory(symbolList, granularity, GlobalConstants.historicMarketData_dir, ".csv")
for symbol in symbolList:
    print(symbolCount)
    symbolCount += 1
    #symbolDataframe = Utils.readCsv(symbol, granularity, start_date_to_download, dirname, ".csv",  periodInString)
    symbolDataframe = Utils.readCsv_from_startdate_enddate(symbol, granularity, start_date_to_read, end_date_to_read, dirname, ".csv")
    # from above symbolDataframe only keep the entries which are 6 rows apart from each other
    symbolDataframe = Utils.filter_dataframe_by_days(symbolDataframe, filterTypeDays)
    
    if symbolDataframe.empty:
        continue
    
    symbolDataframe = Utils.convert_yf_datetime_to_pandas_datetime(symbolDataframe)

    Utils.convert_datetime64_to_user_format(symbolDataframe['Date'], '%d-%m-%Y %H:%M:%S')
    symbolDataframe['Time'] = Utils.convert_datetime64_to_user_format(symbolDataframe['Date'], '%H:%M:%S')
    symbolDataframe = Utils.convert_yf_datetime_to_pandas_datetime(symbolDataframe)
    symbolDataframe['E10'] = symbolDataframe['Close'].ewm(span=10, adjust=False).mean()
    symbolDataframe['E21'] = symbolDataframe['Close'].ewm(span=21, adjust=False).mean()
    symbolDataframe['E50'] = symbolDataframe['Close'].ewm(span=50, adjust=False).mean()
    symbolDataframe.dropna(inplace=True)
    symbolDataframe['isCloseAbove10'] = symbolDataframe.apply(lambda x : 1 if x['Close'] > x['E10'] else -1, axis=1)
    symbolDataframe['isCloseAbove21'] = symbolDataframe.apply(lambda x : 1 if x['Close'] > x['E21'] else -1, axis=1)
    symbolDataframe['isCloseAbove50'] = symbolDataframe.apply(lambda x : 1 if x['Close'] > x['E50'] else -1, axis=1)
    symbolDataframe['isE10AboveE21'] = symbolDataframe.apply(lambda x : 1 if x['E10'] > x['E21'] else -1, axis=1)
    symbolDataframe['isE21AboveE50'] = symbolDataframe.apply(lambda x : 1 if x['E21'] > x['E50'] else -1, axis=1)
    symbolDataframe['isE10AboveE50'] = symbolDataframe.apply(lambda x : 1 if x['E10'] > x['E50'] else -1, axis=1)
    symbolDataframe['TotalValue'] = symbolDataframe['isCloseAbove10'] + symbolDataframe['isCloseAbove21'] + \
                                        symbolDataframe['isCloseAbove50'] + symbolDataframe['isE10AboveE21'] + \
                                            symbolDataframe['isE21AboveE50'] + symbolDataframe['isE10AboveE50']
    # ci = CCIIndicator(high=symbolDataframe['High'], low=symbolDataframe['Low'], close=symbolDataframe['Close'], window=3)
    # symbolDataframe['choppiness'] = ci.cci()
    # atr = AverageTrueRange(high=symbolDataframe['High'], low=symbolDataframe['Low'], close=symbolDataframe['Close'], window=2).average_true_range()
    # sum_atr = atr.rolling(window=2).sum()
    # high_max = symbolDataframe['High'].rolling(window=2).max()
    # low_min = symbolDataframe['Low'].rolling(window=2).min()
    # ci = 100 * np.log10(sum_atr / (high_max - low_min)) / np.log10(2)
    # symbolDataframe['choppiness'] = ci
    symbolDataframe['Symbol'] = symbol
    symbolDataframe['ReturnChange'] = ((symbolDataframe['Close'].shift(3) - symbolDataframe['Close'].shift(23))/symbolDataframe['Close'].shift(23)) * 100
    symbolDataframe.dropna(inplace=True)
    symbolDataframe['PRET'] = symbolDataframe.apply(lambda x : 1 if x['ReturnChange'] > 0 else -1, axis=1)
    symbolDataframe['Direction'] = symbolDataframe['Close'].diff()
    symbolDataframe['PositivePercentage'] = symbolDataframe['Direction'].shift(3).rolling(window=30).apply(lambda x: len([+1 for i in x if i > 0]))
    symbolDataframe['NegativePercentage'] = symbolDataframe['Direction'].shift(3).rolling(window=30).apply(lambda x: len([+1 for i in x if i < 0]))
    symbolDataframe['NoChange'] = symbolDataframe['Direction'].shift(3).rolling(window=30).apply(lambda x: len([+1 for i in x if i == 0]))
    #symbolDataframe['Sector'] = symbolToSectorDict[symbol]
    symbolDataframe['FIPValue1'] = symbolDataframe['PRET'] * ((symbolDataframe['NegativePercentage']/30)- (symbolDataframe['PositivePercentage']/30))
    # for the above symbolDataframe create a new column called FIPValue whose value is symbolDataframe['NegativePercentage']/57 if PRET - -1 else symbolDataframe['PositivePercentage']/57
    symbolDataframe['FIPValue'] = symbolDataframe.apply(lambda x : x['NegativePercentage']/20 if x['PRET'] == -1 else x['PositivePercentage']/20, axis=1)
    #print(symbolDataframe.tail(50).loc[:, ['Close', 'Direction',  'PRET', 'PositivePercentage', 'NegativePercentage', 'FIPValue']])
    if symbolDataframe.empty:
        continue
    
    allSymbologyDataFrame.append(symbolDataframe)

for i in range(allSymbologyDataFrame.__len__()):
    allSymbologyDataFrame[i] = allSymbologyDataFrame[i][~allSymbologyDataFrame[i]['FIPValue'].isna()]
    allSymbologyDataFrame[i].reset_index(drop=True, inplace=True)

dates = allSymbologyDataFrame[0].iloc[0:]['Date'].to_list()
# AllDates = [date.strftime('%d-%m-%Y') for date in dates]
# AllDates
for i in range(allSymbologyDataFrame.__len__()):
    #allSymbologyDataFrame[i]['StrDate'] = allSymbologyDataFrame[i]['Date'].dt.strftime('%Y-%m-%d').copy()
    # Use .loc to avoid the warning about setting a value on a copy of a slice
    allSymbologyDataFrame[i].loc[:, 'StrDate'] = allSymbologyDataFrame[i]['Date'].dt.strftime('%Y-%m-%d').copy()
    
    allSymbologyDataFrame[i].reset_index(drop=True, inplace=True)


# For allSymbologyDataFrame at index 0, get the row with the Date value as 'YYYY-MM-DD' format
date_to_find = '2021-01-01'
AllDates = [date.strftime('%Y-%m-%d') for date in dates]
print(AllDates)
totalDatesToRun = AllDates.__len__()
startDateForRun = '2014-12-10'
endDateTillRun = '2027-03-27'
forwardValue = 1 # indicates how many candles to move forward by....
totalScoreFilter = 5
# Find the next nearest startDateForRun entry value in AllDates and its index
nearest_start_date_index = next((i for i, date in enumerate(AllDates) if date >= startDateForRun), None)
nearest_start_date = AllDates[nearest_start_date_index] if nearest_start_date_index is not None else None

nearest_end_date_index = next((i for i, date in enumerate(AllDates) if date >= endDateTillRun), None)
nearest_end_date = AllDates[nearest_end_date_index] if nearest_end_date_index is not None else None

if nearest_end_date_index is None:
    nearest_end_date_index = totalDatesToRun - 1
    nearest_end_date = AllDates[nearest_end_date_index]
print(nearest_start_date)
print(nearest_end_date)
cashInPortfolio = 1000000
portfolioList = []
symbolToPriceDict = {}
symbolPurchasedToQtyDict = {}
weeklyPortfolioValueDf = pd.DataFrame(columns=['Date', 'AccountValue'])
AdvanceDeclineDf = pd.DataFrame(columns=['Date', 'TotalAdvance'])


def getQtyPurchasedForSymbol(symbol):
    if symbolPurchasedToQtyDict.get(symbol) is None:
        return 0
    return symbolPurchasedToQtyDict[symbol]

def get_portfolio_value():
    totalPortfolioValue = 0
    for symbol in portfolioList:
        totalPortfolioValue += symbolToPriceDict[symbol] * getQtyPurchasedForSymbol(symbol)
    return totalPortfolioValue

def get_account_value():
    return (get_portfolio_value() + cashInPortfolio)

for runDate in range(nearest_start_date_index, nearest_end_date_index-1, forwardValue):
    current_run_date = AllDates[runDate]
    next_run_date = AllDates[runDate + 1]
    portfolioSymbolsToAdd = []
    currentSymbolToPriceDict= {}
    choppinessList = []
    advanceDecline = 0
    symbolDf1 = pd.DataFrame(columns=['Symbol', 'TotalScore', 'FIPScore'])

    for symbolDf  in allSymbologyDataFrame:
        filtered_df = Utils.filter_panda_df_in_range_constraint(symbolDf, 'StrDate', current_run_date, current_run_date)
                
        if filtered_df.empty:
            continue
        #choppinessList.append(filtered_df['choppiness'].iloc[0])
        if filtered_df['Direction'].iloc[0] > 0:
            advanceDecline += 1
        
        symbolName = filtered_df['Symbol'].iloc[0]
        currentSymbolPrice = filtered_df['Close'].iloc[0]
        currentSymbolToPriceDict[symbolName] = currentSymbolPrice
        
        meanTotalScoreValue = filtered_df['TotalValue'].mean()
        
        FIPValue = filtered_df['FIPValue'].iloc[0]
        FIPValue1 = filtered_df['FIPValue1'].iloc[0]

        # append the symbolDf above to columns 'Symbol' mapped to symbolName, TotalScore to meanTotalScoreValue and FIPValue to FIPValue1
        dfx = pd.DataFrame({
            'Symbol': [symbolName],
            'TotalScore': [meanTotalScoreValue],
            'FIPScore' : [FIPValue1]
        })
        
        #symbolDf1 = pd.concat([symbolDf1, dfx], ignore_index=True)
        # Ensure that the DataFrame being concatenated is not empty
        # if not dfx.empty and not dfx.isna().all(axis=None):
        #     symbolDf1 = pd.concat([symbolDf1, dfx], ignore_index=True)
            # Ensure that the DataFrame being concatenated is not empty and does not contain all-NA rows
        if symbolDf1.empty:
            symbolDf1 = dfx
        else:
            if not dfx.empty and not dfx.isna().all(axis=None):
                symbolDf1 = pd.concat([symbolDf1, dfx.dropna(how='all')], ignore_index=True)

    advanceDeclineDfx = pd.DataFrame({
            'Date': [current_run_date],
            'TotalAdvance' : [(advanceDecline/224) * 100]
    })
    
    if AdvanceDeclineDf.empty:
        AdvanceDeclineDf = advanceDeclineDfx
    else:
        if not advanceDeclineDfx.empty and not advanceDeclineDfx.isna().all(axis=None):
            AdvanceDeclineDf = pd.concat([AdvanceDeclineDf, advanceDeclineDfx.dropna(how='all')], ignore_index=True)
    
    #DatesList = nifyvix['Date'].to_list()
    #nearest_next_index = next((i for i, date in enumerate(DatesList) if date >= current_run_date), None) - 1
    
    #print("nifyvix date", nifyvix.iloc[nearest_next_index]['Date'])
    
    #print("weekly date {} choppiness value {}".format(current_run_date, mean_choppiness_value))
    # sort symbolDf1 first based on TotalScore descending then sort on FIPScore ascending
    symbolDf1 = symbolDf1.sort_values(by=['TotalScore', 'FIPScore'], ascending=[False, True])

    # from above symbolDf1 get the top 10 symbolNames
    #print("nearest_next_index", nearest_next_index)
    # if nifyvix.iloc[nearest_next_index]['percentile'] < 66:
    portfolioSymbolsToAdd = symbolDf1.head(6)['Symbol'].tolist()
    # else:
    #     portfolioSymbolsToAdd = []
    #portfolioSymbolsToAdd.extend(top_symbols)

    # find common elements between portfolioList and portfolioSymbolsToAdd 
    commonSymbols = list(set(portfolioList) & set(portfolioSymbolsToAdd))
    symbolsToLiquidate = list(set(portfolioList) - set(commonSymbols))
    symbolsToAdd = list(set(portfolioSymbolsToAdd) - set(commonSymbols))
            
    for symbol in symbolsToLiquidate:
        cashInPortfolio += currentSymbolToPriceDict[symbol] * getQtyPurchasedForSymbol(symbol)
        cashInPortfolio -= currentSymbolToPriceDict[symbol] * getQtyPurchasedForSymbol(symbol) * 0.001
        portfolioList.remove(symbol)
    

    if symbolsToAdd.__len__() > 0:
        totalSymbols = len(symbolsToAdd)
        cashPerStock = cashInPortfolio/len(symbolsToAdd)
        # if cashPerStock > 5000:
        #     cashPerStock = 5000
        for symbol in symbolsToAdd:
            if totalSymbols <= 0:
                break
            cashPerStock = cashInPortfolio/totalSymbols
            stockQty = cashPerStock/currentSymbolToPriceDict[symbol]
            symbolPurchasedToQtyDict[symbol] = stockQty
            cashInPortfolio -= cashPerStock
            cashInPortfolio -= cashPerStock * 0.001
            totalSymbols -= 1                
            portfolioList.append(symbol)

    symbolToPriceDict = currentSymbolToPriceDict
    df2 = pd.DataFrame({
        'Date': [current_run_date],
        'AccountValue': [get_account_value()]
    })
    if weeklyPortfolioValueDf.empty:
        weeklyPortfolioValueDf = df2
    else:
        weeklyPortfolioValueDf = pd.concat([weeklyPortfolioValueDf, df2], ignore_index=True)
    print(portfolioSymbolsToAdd)
    #print(nifyvix.iloc[nearest_next_index]['percentile'])
    print("Weekly date [" + current_run_date + "] " +  str(get_account_value()))


In [ ]:
print(AllDates)
totalDatesToRun = AllDates.__len__()
startDateForRun = '2014-12-10'
endDateTillRun = '2027-03-27'
forwardValue = 1 # indicates how many candles to move forward by....
totalScoreFilter = 5
# Find the next nearest startDateForRun entry value in AllDates and its index
nearest_start_date_index = next((i for i, date in enumerate(AllDates) if date >= startDateForRun), None)
nearest_start_date = AllDates[nearest_start_date_index] if nearest_start_date_index is not None else None

nearest_end_date_index = next((i for i, date in enumerate(AllDates) if date >= endDateTillRun), None)
nearest_end_date = AllDates[nearest_end_date_index] if nearest_end_date_index is not None else None

if nearest_end_date_index is None:
    nearest_end_date_index = totalDatesToRun - 1
    nearest_end_date = AllDates[nearest_end_date_index]
print(nearest_start_date)
print(nearest_end_date)
cashInPortfolio = 1000000
portfolioList = []
symbolToPriceDict = {}
symbolPurchasedToQtyDict = {}
weeklyPortfolioValueDf = pd.DataFrame(columns=['Date', 'AccountValue', 'StockList'])
AdvanceDeclineDf = pd.DataFrame(columns=['Date', 'TotalAdvance'])


def getQtyPurchasedForSymbol(symbol):
    if symbolPurchasedToQtyDict.get(symbol) is None:
        return 0
    return symbolPurchasedToQtyDict[symbol]

def get_portfolio_value():
    totalPortfolioValue = 0
    for symbol in portfolioList:
        totalPortfolioValue += symbolToPriceDict[symbol] * getQtyPurchasedForSymbol(symbol)
    return totalPortfolioValue

def get_account_value():
    return (get_portfolio_value() + cashInPortfolio)

for runDate in range(nearest_start_date_index, nearest_end_date_index+1, forwardValue):
    current_run_date = AllDates[runDate]
    # next_run_date = AllDates[runDate + 1]
    # print(current_run_date)
    # print(next_run_date)
    portfolioSymbolsToAdd = []
    currentSymbolToPriceDict= {}
    choppinessList = []
    advanceDecline = 0
    symbolDf1 = pd.DataFrame(columns=['Symbol', 'TotalScore', 'FIPScore'])

    for symbolDf  in allSymbologyDataFrame:
        filtered_df = Utils.filter_panda_df_in_range_constraint(symbolDf, 'StrDate', current_run_date, current_run_date)
                
        if filtered_df.empty:
            continue
        #choppinessList.append(filtered_df['choppiness'].iloc[0])
        if filtered_df['Direction'].iloc[0] > 0:
            advanceDecline += 1
        
        symbolName = filtered_df['Symbol'].iloc[0]
        currentSymbolPrice = filtered_df['Close'].iloc[0]
        currentSymbolToPriceDict[symbolName] = currentSymbolPrice
        
        meanTotalScoreValue = filtered_df['TotalValue'].mean()
        
        FIPValue = filtered_df['FIPValue'].iloc[0]
        FIPValue1 = filtered_df['FIPValue1'].iloc[0]

        # append the symbolDf above to columns 'Symbol' mapped to symbolName, TotalScore to meanTotalScoreValue and FIPValue to FIPValue1
        dfx = pd.DataFrame({
            'Symbol': [symbolName],
            'TotalScore': [meanTotalScoreValue],
            'FIPScore' : [FIPValue1]
        })
        
        #symbolDf1 = pd.concat([symbolDf1, dfx], ignore_index=True)
        # Ensure that the DataFrame being concatenated is not empty
        # if not dfx.empty and not dfx.isna().all(axis=None):
        #     symbolDf1 = pd.concat([symbolDf1, dfx], ignore_index=True)
            # Ensure that the DataFrame being concatenated is not empty and does not contain all-NA rows
        if symbolDf1.empty:
            symbolDf1 = dfx
        else:
            if not dfx.empty and not dfx.isna().all(axis=None):
                symbolDf1 = pd.concat([symbolDf1, dfx.dropna(how='all')], ignore_index=True)

    advanceDeclineDfx = pd.DataFrame({
            'Date': [current_run_date],
            'TotalAdvance' : [(advanceDecline/224) * 100]
    })
    
    if AdvanceDeclineDf.empty:
        AdvanceDeclineDf = advanceDeclineDfx
    else:
        if not advanceDeclineDfx.empty and not advanceDeclineDfx.isna().all(axis=None):
            AdvanceDeclineDf = pd.concat([AdvanceDeclineDf, advanceDeclineDfx.dropna(how='all')], ignore_index=True)
    
    #DatesList = nifyvix['Date'].to_list()
    #nearest_next_index = next((i for i, date in enumerate(DatesList) if date >= current_run_date), None) - 1
    
    #print("nifyvix date", nifyvix.iloc[nearest_next_index]['Date'])
    
    #print("weekly date {} choppiness value {}".format(current_run_date, mean_choppiness_value))
    # sort symbolDf1 first based on TotalScore descending then sort on FIPScore ascending
    symbolDf1 = symbolDf1.sort_values(by=['TotalScore', 'FIPScore'], ascending=[False, True])

    # from above symbolDf1 get the top 10 symbolNames
    #print("nearest_next_index", nearest_next_index)
    # if nifyvix.iloc[nearest_next_index]['percentile'] < 66:
    portfolioSymbolsToAdd = symbolDf1.head(6)['Symbol'].tolist()
    # else:
    #     portfolioSymbolsToAdd = []
    #portfolioSymbolsToAdd.extend(top_symbols)

    # find common elements between portfolioList and portfolioSymbolsToAdd 
    commonSymbols = list(set(portfolioList) & set(portfolioSymbolsToAdd))
    symbolsToLiquidate = list(set(portfolioList) - set(commonSymbols))
    symbolsToAdd = list(set(portfolioSymbolsToAdd) - set(commonSymbols))
            
    for symbol in symbolsToLiquidate:
        cashInPortfolio += currentSymbolToPriceDict[symbol] * getQtyPurchasedForSymbol(symbol)
        cashInPortfolio -= currentSymbolToPriceDict[symbol] * getQtyPurchasedForSymbol(symbol) * 0.001
        portfolioList.remove(symbol)
    

    if symbolsToAdd.__len__() > 0:
        totalSymbols = len(symbolsToAdd)
        cashPerStock = cashInPortfolio/len(symbolsToAdd)
        # if cashPerStock > 5000:
        #     cashPerStock = 5000
        for symbol in symbolsToAdd:
            if totalSymbols <= 0:
                break
            cashPerStock = cashInPortfolio/totalSymbols
            stockQty = cashPerStock/currentSymbolToPriceDict[symbol]
            symbolPurchasedToQtyDict[symbol] = stockQty
            cashInPortfolio -= cashPerStock
            cashInPortfolio -= cashPerStock * 0.001
            totalSymbols -= 1                
            portfolioList.append(symbol)

    symbolToPriceDict = currentSymbolToPriceDict
    df2 = pd.DataFrame({
        'Date': [current_run_date],
        'AccountValue': [get_account_value()],
        'StockList': [portfolioSymbolsToAdd]
    })
    if weeklyPortfolioValueDf.empty:
        weeklyPortfolioValueDf = df2
    else:
        weeklyPortfolioValueDf = pd.concat([weeklyPortfolioValueDf, df2], ignore_index=True)
    print(portfolioSymbolsToAdd)
    #print(nifyvix.iloc[nearest_next_index]['percentile'])
    print("Weekly date [" + current_run_date + "] " +  str(get_account_value()))


url = "https://swagger.bling-trading.com/BasketOrderLeg/Create"
params = {
    "portfolioId": "14",
    "strategyId": "1"
}

headers = {
    "Accept": "*/*",
    "Accept-Encoding": "gzip, deflate, br, zstd",
    "Accept-Language": "en-US,en;q=0.9",
    "Connection": "keep-alive",
    "Origin": "https://gui.bling-trading.com",
    "Referer": "https://gui.bling-trading.com/portfolio/14/1",
    "Sec-Fetch-Dest": "empty",
    "Sec-Fetch-Mode": "cors",
    "Sec-Fetch-Site": "same-site",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/134.0.0.0 Safari/537.36",
    "Content-Type": "application/json",
    "sec-ch-ua": '"Chromium";v="134", "Not:A-Brand";v="24", "Google Chrome";v="134"',
    "sec-ch-ua-mobile": "?0",
    "sec-ch-ua-platform": '"Windows"',
    "x-bling-token": "g4CjqsMDK-lfxwG",
    "x-client-domain": "BLING",
    "x-platform": "WEB_SITE",
    "x-referrer-domain": "BLING"
}

data = {
    "StockName": "DIVISLAB",
    "Side": "Buy",
    "WeightPercentage": 2,
    "ProfitTakingPercentage": None,
    "StopLossPercentage": None,
    "IsTrailingStopLoss": None
}

for symbol in weeklyPortfolioValueDf.tail(1)['StockList'].values[0]:
    data['StockName'] = symbol
        # Make the POST request
    response = requests.post(url, headers=headers, params=params, json=data)
    print("Status Code:", response.status_code)
    print("Response:", response.text)





In [ ]:
import requests
url = "https://swagger.bling-trading.com/BasketOrderLeg/Create"
params = {
    "portfolioId": "14",
    "strategyId": "1"
}

headers = {
    "Accept": "*/*",
    "Accept-Encoding": "gzip, deflate, br, zstd",
    "Accept-Language": "en-US,en;q=0.9",
    "Connection": "keep-alive",
    "Origin": "https://gui.bling-trading.com",
    "Referer": "https://gui.bling-trading.com/portfolio/14/1",
    "Sec-Fetch-Dest": "empty",
    "Sec-Fetch-Mode": "cors",
    "Sec-Fetch-Site": "same-site",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/134.0.0.0 Safari/537.36",
    "Content-Type": "application/json",
    "sec-ch-ua": '"Chromium";v="134", "Not:A-Brand";v="24", "Google Chrome";v="134"',
    "sec-ch-ua-mobile": "?0",
    "sec-ch-ua-platform": '"Windows"',
    "x-bling-token": "g4CjqsMDK-lfxwG",
    "x-client-domain": "BLING",
    "x-platform": "WEB_SITE",
    "x-referrer-domain": "BLING"
}

data = {
    "StockName": "DIVISLAB",
    "Side": "Buy",
    "WeightPercentage": 2,
    "ProfitTakingPercentage": None,
    "StopLossPercentage": None,
    "IsTrailingStopLoss": None
}

for symbol in weeklyPortfolioValueDf.tail(1)['StockList'].values[0]:
    data['StockName'] = symbol
        # Make the POST request
    response = requests.post(url, headers=headers, params=params, json=data)
    print("Status Code:", response.status_code)
    print("Response:", response.text)
